1. BaseTool — 추상 기본 클래스 (설계도)
2. 구체적인 Tool 3개 — Calculator, StringProcessor, Dictionary
3. ToolRegistry — 도구 보관함
4. Tool 체이닝 — 도구를 연결해서 실행
5. ReAct 시뮬레이션 통합 — 전부 합쳐서 실행

In [7]:
# BaseTool 추상 클래스
from abc import ABC, abstractmethod
from typing import Dict, Optional
# abc : Abstract Base Class 모듈
# ABC : 추상 클래스를 만들기 위해 상속받는 클래스
# abstractmethod : "이 메서드 반드시 구현해야 해!"라고 강제하는 데코레이터

class BaseTool(ABC):    # ABC 상속 -> 추상 클래스가 됨
    """ReAct 에이전트에서 사용하는 도구의 기본 클래스"""

    def __init__(self, name: str, description: str):
        """
        모든 도구가 공통으로 가져야할 속성
        name : 도구 이름
        description : LLM이 읽고 언제 쓸지 판단하는 설명
        """
        self.name = name
        self.description = description

    @abstractmethod
    def execute(self, input_text: str) -> str:
        # @abstractmethod : 이 메서드는 반드시 하위 클래스에서 구현해야 함
        # 모든 도구가 반드시 execute()를 갖도록 강제하는 장치
        pass    # 여기선 내용 없음. 하위 클래스가 채워야 함

    def get_schema(self) -> Dict:
        """
        도구의 정보를 딕셔너리로 반환
        """
        return {'type': 'string', 'description': '입력텍스트'}
    
    def __repr__(self):
        # 객체를 print()할 때 보여줄 문자열 정의
        # 없으면 → <__main__.CalculatorTool object at 0x...> 처럼 못생기게 나옴
        # 있으면 → Tool(name=Calculator) 처럼 깔끔하게 나옴
        return f'Tool(name={self.name})'

In [8]:
# 구체적인 Tool 구현체 3개
class CalculatorTool(BaseTool):
    """수학 수식을 계산하는 도구"""

    def __init__(self):
        super().__init__(
            name='Calculator',
            description='수학 수식을 계산합니다. 사칙연산 거듭제곱 나머지연산 등을 지원합니다.'
        )

    def execute(self, input_text: str) -> str:
        # @abstractmethod 였던 execute()를 여기서 구현
        # input_text : LLM이 Action에서 넘긴 값
        try:
            # 보안처리
            # eval()은 문자열을 그대로 실행하는 함수라 위험할 수 있음
            # 그래서 허용된 문자만 통과시키는 필터링을 먼저 수행
            allowed_chars = set('0123456789+-*/.() ')

            if not all(c in allowed_chars for c in input_text):
            # all() : 모두 True여야 True. 하나라도 False면 False
                return f"오류: 허용되지 않는 문자가 포함되어 있습니다."
            
            result = eval(input_text)
            return f"계산 결과: {result}"
        except Exception as e:
            return f'계산 오류: {str(e)}'
        

In [9]:
class StringProcessorTool(BaseTool):
    """문자열 처리 도구"""

    def __init__(self):
        super().__init__(
            name="StringProcessor",
            description="문자열의 길이, 단어 수, 글자 수 등을 분석합니다."
        )

    def execute(self, input_text: str) -> str:
        char_count = len(input_text)
        # len() : 문자열 전체 길이 (공백 포함)
        # 예) "ReAct는 대세입니다." → 13

        word_count = len(input_text.split())
        # .split() : 공백 기준으로 쪼갬 → 리스트 반환
        # 예) "ReAct는 대세입니다.".split() → ["ReAct는", "대세입니다."]
        # len() → 2

        line_count = len(input_text.splitlines())
        # .splitlines() : 줄바꿈(\n) 기준으로 쪼갬
        # 예) "안녕\n반가워".splitlines() → ["안녕", "반가워"] → 2줄

        return (f"문자 수: {char_count}, 단어 수: {word_count}, "
                f"줄 수: {line_count}")
        # 괄호로 묶어서 두 줄에 나눠 쓴 것 (문법적으로 한 줄과 동일)

In [10]:
class DictionaryTool(BaseTool):
    """미리 정의된 사전에서 정보를 검색하는 도구"""

    def __init__(self):
        super().__init__(
            name="Dictionary",
            description="내장 사전에서 용어의 정의를 검색합니다."
        )
        # 사전 데이터를 딕셔너리로 정의
        # → 실제 서비스라면 DB나 외부 API에서 가져오겠지만
        #   여기선 시뮬레이션이니까 하드코딩
        self.entries = {
            "인공지능": "인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하는 기술",
            "머신러닝": "데이터로부터 패턴을 학습하여 예측이나 결정을 수행하는 AI의 하위 분야",
            "딥러닝": "다층 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야",
            "LLM": "대규모 텍스트 데이터로 사전 학습된 거대 언어 모델",
            "ReAct": "추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법"
            # ... 생략
        }

    def execute(self, input_text: str) -> str:
        term = input_text.strip()
        # .strip() : 앞뒤 공백 제거
        # 예) "  ReAct  " → "ReAct"

        # ── 1단계: 정확히 일치하는 항목 검색 ──────────────
        if term in self.entries:
            return f"{term}: {self.entries[term]}"
            # 예) "ReAct" → "ReAct: 추론(Reasoning)과 행동(Acting)을..."

        # ── 2단계: 부분 일치 검색 (정확히 없으면) ──────────
        matches = [k for k in self.entries if term in k or k in term]
        # 리스트 컴프리헨션으로 부분 일치 항목 수집
        # term in k : 검색어가 키에 포함되는지  예) "러닝" in "머신러닝" → True
        # k in term : 키가 검색어에 포함되는지  예) "LLM" in "LLM 설명" → True
        # 예) term="ReAct는 대세입니다. 트랜스포머"
        #     → "트랜스포머" in term → True → matches에 포함

        if matches:
            results = [f"{k}: {self.entries[k]}" for k in matches]
            # 매칭된 항목들을 "키: 값" 형식으로 변환
            return "관련 항목:\n" + "\n".join(results)
            # "\n".join() : 리스트를 줄바꿈으로 연결해서 하나의 문자열로

        return f"'{term}'에 대한 정보를 찾을 수 없습니다."
        # 완전히 못 찾은 경우

In [13]:
# ToolRegistry class
class ToolRegistry:
    """도구를 등록하고 관리하는 레지스트리"""

    def __init__(self):
        self._tools: Dict[str, BaseTool] = {}
        # _tools : 도구들을 저장하는 딕셔너리
        # 언더스코어(_) : 내부용 변수. 외부에서 직접 접근 비권장
        # Dict[str, BaseTool] : 타입 힌트
        #   → 키는 str (도구 이름), 값은 BaseTool (도구 객체)
        #   → 예) {"Calculator": CalculatorTool객체, "Dictionary": DictionaryTool객체}
        # {} : 처음엔 빈 딕셔너리로 시작

    def register(self, tool: BaseTool) -> 'ToolRegistry':
        # 도구를 보관함에 등록하는 메서드
        # tool : BaseTool을 상속한 도구 객체 (Calculator, Dictionary 등)

        self._tools[tool.name] = tool
        # tool.name을 키로, tool 객체 자체를 값으로 저장
        # 예) tool = CalculatorTool()
        #     tool.name = "Calculator"
        #     → self._tools["Calculator"] = CalculatorTool객체

        return self
        # 메서드 체이닝을 위해 자기 자신 반환
        # → registry.register(CalculatorTool()).register(DictionaryTool()) 가능
    
    def get(self, name: str) -> Optional[BaseTool]:
        # 이름으로 도구 객체를 꺼내오는 메서드
        # Optional[BaseTool] : BaseTool 객체 또는 None 반환 가능

        return self._tools.get(name)
        # 딕셔너리의 .get() : 키가 없으면 None 반환 (에러 안 남)
        # 예) self._tools.get("Calculator") → CalculatorTool 객체
        #     self._tools.get("없는도구")   → None
        # 참고) self._tools["없는도구"] 로 쓰면 KeyError 발생 → .get()이 안전
    
    def execute(self, tool_name: str, input_text: str) -> str:
        # 도구 이름과 입력값을 받아서 실행까지 해주는 핵심 메서드
        # → LLM이 "Action: Calculator[(100+250)*0.1]" 을 뱉으면
        #   Parser가 tool_name="Calculator", input_text="(100+250)*0.1" 을 추출하고
        #   이 메서드를 호출해서 실제 실행

        tool = self.get(tool_name)
        # 이름으로 도구 객체 조회

        if tool is None:
            # 등록되지 않은 도구를 LLM이 요청한 경우
            # → 에러를 터뜨리지 않고 문자열로 반환
            #   왜? Observation으로 LLM에게 다시 전달해야 하니까!
            #   LLM이 이 메시지를 보고 "아 그 도구는 없구나" 하고 재추론
            return f'오류 : {tool_name} 도구를 찾을수 없습니다. 사용가능한 도구 : {self._tools.keys()}'

        try:
            return tool.execute(input_text)
            # 도구 객체의 execute() 호출
            # → CalculatorTool.execute("(100+250)*0.1") → "계산 결과: 35.0"
        except Exception as e:
            return f'실행 오류 : {e}'
            # execute() 내부에서 예상치 못한 에러 발생 시 처리
    
    def list_tools(self) -> str:
        # 등록된 모든 도구의 이름과 설명을 문자열로 반환
        # → 프롬프트에 "사용 가능한 도구 목록" 넣을 때 사용
        descriptions = []
        for name, tool in self._tools.items():
            # .items() : 딕셔너리를 (키, 값) 쌍으로 순회
            # name = "Calculator", tool = CalculatorTool 객체
            descriptions.append(f'  - {name} : {tool.description}')
        return "사용가능한도구\n" + "\n".join(descriptions)
        # → "사용가능한도구
        #     - Calculator : 수학 수식을 계산합니다...
        #     - Dictionary : 내장 사전에서..."
    
    def get_all_schemas(self) -> list:
        # 모든 도구의 스키마를 리스트로 반환
        # → LLM API에 도구 목록을 전달할 때 사용 (Function Calling 등)
        return [tool.get_schema() for tool in self._tools.values()]
        # .values() : 딕셔너리에서 값(도구 객체)만 순회
        # 각 도구의 get_schema() 호출해서 리스트로 수집

In [14]:
# ── 실제 사용 예시 ───────────────────────────────────────────

registry = ToolRegistry()         # 빈 보관함 생성

registry.register(CalculatorTool())      # Calculator 등록
registry.register(StringProcessorTool()) # StringProcessor 등록
registry.register(DictionaryTool())      # Dictionary 등록
# → 내부적으로 self._tools = {
#       "Calculator": CalculatorTool객체,
#       "StringProcessor": StringProcessorTool객체,
#       "Dictionary": DictionaryTool객체
#   }

print(registry.execute("Calculator", "2 ** 10"))
# → get("Calculator") → CalculatorTool객체
# → CalculatorTool.execute("2 ** 10")
# → "계산 결과: 1024"

print(registry.execute("UnknownTool", "test"))
# → get("UnknownTool") → None
# → "오류 : UnknownTool 도구를 찾을수 없습니다..."

계산 결과: 1024
오류 : UnknownTool 도구를 찾을수 없습니다. 사용가능한 도구 : dict_keys(['Calculator', 'StringProcessor', 'Dictionary'])


In [15]:
# Tool chaining 
def tool_chain(registry, steps):
    """여러 도구를 순차적으로 실행하는 체이닝 함수

    Args:
        registry : ToolRegistry 인스턴스 (도구 보관함)
        steps    : (tool_name, input_data) 튜플의 리스트
                   input_data가 callable이면 이전 결과를 인자로 받음
    """
    print("=== Tool 체이닝 실행 ===")
    previous_result = None
    # 이전 단계의 실행 결과를 저장하는 변수
    # 처음엔 아직 결과가 없으니 None으로 초기화
    # → 다음 단계가 이전 결과를 입력으로 쓸 수 있게 연결해주는 핵심 변수

    for i, (tool_name, input_data) in enumerate(steps, 1):
        # enumerate(steps, 1) : steps 리스트를 순회하면서 1부터 번호 부여
        # (tool_name, input_data) : 튜플 언패킹
        #   → steps의 각 항목이 ("Calculator", "(100+200)/3") 처럼 튜플이니까
        #     tool_name = "Calculator", input_data = "(100+200)/3" 으로 분리

        # ── 핵심 분기 ──────────────────────────────────────
        if callable(input_data):
            actual_input = input_data(previous_result)
            # callable() : 해당 값이 호출 가능한지 (함수인지) 확인
            # → input_data가 lambda prev: prev 같은 함수면 True
            # → 이전 결과(previous_result)를 인자로 넘겨서 실제 입력값 생성
            # 예) lambda prev: prev → 이전 결과를 그대로 다음 입력으로 사용
        else:
            actual_input = input_data
            # input_data가 그냥 문자열이면 그대로 사용
            # 예) "LLM", "(100+200)/3" 같은 고정 입력값

        print(f"\n[Chain Step {i}] {tool_name}")
        print(f"  입력: {actual_input}")

        result = registry.execute(tool_name, actual_input)
        # Registry를 통해 도구 실행
        # → tool_name으로 도구 찾고 actual_input으로 실행

        print(f"  출력: {result}")
        previous_result = result
        # 이번 결과를 previous_result에 저장
        # → 다음 루프에서 callable인 경우 이 값을 입력으로 사용

    print(f"\n최종 결과: {previous_result}")
    return previous_result

In [16]:
# ── 체이닝 예시 1: 계산 → 문자열 분석 ─────────────────────

chain_steps = [
    ("Calculator", "(100 + 200 + 300) / 3"),
    # 1단계: 고정 입력값 → "(100 + 200 + 300) / 3" 그대로 사용
    # → 실행 결과: "계산 결과: 200.0"

    ("StringProcessor", lambda prev: prev),
    # 2단계: lambda prev: prev → callable!
    # → previous_result = "계산 결과: 200.0" 을 그대로 입력으로 사용
    # → StringProcessor.execute("계산 결과: 200.0")
    # → 실행 결과: "문자 수: 12, 단어 수: 3, 줄 수: 1"
]
tool_chain(registry, chain_steps)

=== Tool 체이닝 실행 ===

[Chain Step 1] Calculator
  입력: (100 + 200 + 300) / 3
  출력: 계산 결과: 200.0

[Chain Step 2] StringProcessor
  입력: 계산 결과: 200.0
  출력: 문자 수: 12, 단어 수: 3, 줄 수: 1

최종 결과: 문자 수: 12, 단어 수: 3, 줄 수: 1


'문자 수: 12, 단어 수: 3, 줄 수: 1'

In [17]:
# ── 체이닝 예시 2: 사전 검색 → 문자열 분석 → 계산 ─────────

chain_steps_2 = [
    ("Dictionary", "LLM"),
    # 1단계: 고정 입력 "LLM" 으로 사전 검색
    # → "LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델"

    ("StringProcessor", lambda prev: prev),
    # 2단계: 1단계 결과를 그대로 입력으로 사용
    # → StringProcessor.execute("LLM: 대규모 텍스트...")
    # → "문자 수: 33, 단어 수: 9, 줄 수: 1"

    ("Calculator", "7 * 8 + 6"),
    # 3단계: 고정 입력 (이전 결과와 무관하게 독립적으로 실행)
    # → callable이 아닌 고정 문자열이라 previous_result 무시
    # → "계산 결과: 62"
]
tool_chain(registry, chain_steps_2)

=== Tool 체이닝 실행 ===

[Chain Step 1] Dictionary
  입력: LLM
  출력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델

[Chain Step 2] StringProcessor
  입력: LLM: 대규모 텍스트 데이터로 사전 학습된 거대 언어 모델
  출력: 문자 수: 33, 단어 수: 9, 줄 수: 1

[Chain Step 3] Calculator
  입력: 7 * 8 + 6
  출력: 계산 결과: 62

최종 결과: 계산 결과: 62


'계산 결과: 62'

In [18]:
# 지금 코드 (수동 체이닝 - 개발자가 순서 직접 지정)
chain_steps = [
    ("Calculator", "(100+200+300)/3"),
    ("StringProcessor", lambda prev: prev),
]

# 실제 ReAct Agent (자동 체이닝 - LLM이 순서 스스로 결정)
# Thought: 먼저 계산해야 합니다
# Action: Calculator[(100+200+300)/3]
# Observation: 계산 결과: 200.0
# Thought: 이 결과의 문자 수를 세어야 합니다
# Action: StringProcessor[계산 결과: 200.0]  ← LLM이 알아서 이전 결과를 입력으로 씀

In [19]:
# ReAct 시뮬레이션 통합
def simulate_react_with_tools(question, steps, registry):
    """Tool Registry를 사용한 ReAct 시뮬레이션
    
    Args:
        question : 풀어야 할 질문
        steps    : 미리 정의된 Thought-Action 시퀀스 (딕셔너리 리스트)
        registry : ToolRegistry 인스턴스
    """
    print(f"Question: {question}")
    print("=" * 50)

    for i, step in enumerate(steps, 1):
        # steps 리스트를 1번부터 순회
        # step : 딕셔너리 하나
        # → {"thought": "...", "action": "Calculator", "input": "(100+250)*0.1"}
        print(f"\n--- Step {i} ---")
        print(f"Thought: {step['thought']}")
        # step['thought'] : 이번 단계의 추론 내용 출력

        # ── 핵심 분기: Finish인지 아닌지 ──────────────────
        if step['action'] == 'Finish':
            # Finish 액션 = 더 이상 도구 실행 없이 최종 답변 출력하고 종료
            print(f"Action: Finish[{step['input']}]")
            print(f"\n{'=' * 50}")
            print(f"Final Answer: {step['input']}")
            return step['input']
            # return으로 함수 즉시 종료
            # → 이후 steps가 더 있어도 실행 안 함

        # ── Finish가 아니면 도구 실행 ──────────────────────
        print(f"Action: {step['action']}[{step['input']}]")
        # 예) "Action: Calculator[(100 + 250) * 0.1]"

        observation = registry.execute(step['action'], step['input'])
        # Registry를 통해 실제 도구 실행
        # step['action'] = "Calculator" → 도구 찾기
        # step['input']  = "(100+250)*0.1" → 도구에 입력
        # → CalculatorTool.execute("(100+250)*0.1") → "계산 결과: 35.0"

        print(f"Observation: {observation}")
        # 도구 실행 결과를 Observation으로 출력
        # → 실제 Agent에서는 이 값이 다시 LLM에게 전달됨

    return None
    # 모든 steps를 다 돌았는데 Finish가 없었던 경우
    # → 정상적으론 발생하면 안 됨 (설계 오류)

In [20]:
# ── 실제 사용 예시 ───────────────────────────────────────────

steps = [
    {
        "thought": "(100 + 250) * 0.1 의 값을 계산해야 합니다.",
        "action": "Calculator",          # Registry에서 이 이름으로 도구 찾음
        "input": "(100 + 250) * 0.1"    # 도구에 넘길 입력값
    },
    {
        "thought": "계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.",
        "action": "Dictionary",
        "input": "ReAct"
    },
    {
        "thought": "필요한 정보를 모두 확인했습니다.",
        "action": "Finish",              # Finish → 도구 실행 없이 바로 종료
        "input": "(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다."
    }
]

simulate_react_with_tools(
    "(100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?",
    steps,
    registry   # 앞에서 만들어둔 Registry (Calculator, Dictionary 등록돼 있음)
)

Question: (100+250)*0.1의 값은 얼마이며, ReAct의 정의는 무엇인가?

--- Step 1 ---
Thought: (100 + 250) * 0.1 의 값을 계산해야 합니다.
Action: Calculator[(100 + 250) * 0.1]
Observation: 계산 결과: 35.0

--- Step 2 ---
Thought: 계산 결과를 확인했습니다. 이제 ReAct에 대한 정의를 찾아보겠습니다.
Action: Dictionary[ReAct]
Observation: ReAct: 추론(Reasoning)과 행동(Acting)을 결합한 프롬프팅 기법

--- Step 3 ---
Thought: 필요한 정보를 모두 확인했습니다.
Action: Finish[(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.]

Final Answer: (100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.


'(100+250)*0.1 = 35.0이고, ReAct는 추론과 행동을 결합한 프롬프팅 기법입니다.'